# Kotoba TTS v1.5 — Promptless Real-time Text-to-Speech (Bidirectional Streaming)

This notebook demonstrates how to use the Kotoba TTS v1.5 (promptless) model package from AWS Marketplace.

**Product**: Kotoba TTS v1.5 — Promptless Real-time Japanese Speech Synthesis  
**Seller**: Kotoba AI

## Key Features

- **One-shot synthesis** — submit the full text inside a single `response.create`; the server streams `audio.chunk` deltas back and finishes with `response.done`
- **Multiple shipped voice presets** — `default` is always available; additional named presets (e.g. `man-1`, `woman-1`) may also ship with the bundle
- **Selectable output format & sample rate** — choose `pcm_f32`, `pcm_16`, `mulaw`, or `opus` per session (PCM at 8 / 16 / 24 kHz, mono); negotiated on `open` and echoed on `session.created`
- **Pipelining & cancellation** — submit several `response.create`s without waiting for each `response.done`: extras are accepted with `response.queued` and drained FIFO, and any **active or queued** response can be cancelled by `response_id` (see §5)
- **Correlated, classified errors** — request errors carry the offending `response_id` and a `fatal` flag (`false` ⇒ keep going, `true` ⇒ reconnect), so a pipelined client knows exactly what failed
- Scalable GPU-accelerated inference

## What's new (versioning)

This is **Kotoba TTS v1.5**. The one-shot wire protocol introduced in v1.0 is unchanged at its core; later releases layered capabilities on top:

- **v1.5 — Pipelining & cancellation.** Send the next `response.create` without waiting: extra requests are accepted with `response.queued` and drain FIFO (`response.done(N)` always precedes `response.created(N+1)`). `response.cancel` takes an optional `response_id` to cancel a specific **active or queued** response. Request errors are now **non-fatal where possible** and carry a `fatal` flag plus the offending `response_id`. See §5.
- **v1.4 — Selectable output audio.** `open` accepts optional `format` (`pcm_f32` / `pcm_16` / `mulaw` / `opus`) and `sample_rate`; the negotiated values are echoed on `session.created`. See §4.2.
- **v1.0 — One-shot protocol.** The full `text` ships inside `response.create`, response-scoped events carry a `response.id` / `response_id` for correlation, and multiple preset voices are selectable via `speaker_id`.

**Migrating from v0.1** (input-streaming): the wire protocol is different. v0.1's `input_text_buffer.append` / `input_text_buffer.commit` events were removed — place the full text on `response.create` instead. The v0.1 protocol reference lives at [docs/tts-v0.1/](../tts-v0.1/).

---

## IMPORTANT: Bidirectional Streaming Only

This product uses **SageMaker bidirectional streaming** exclusively:

- Standard `POST /invocations` is **NOT supported**
- Batch transform jobs are **NOT supported**
- Requires **Python 3.12+** with `aws-sdk-python[sagemaker-runtime-http2]`

---

## Prerequisites

1. **AWS Account** with SageMaker permissions
2. **Python 3.12+** (required for HTTP/2 SDK)
3. **AWS Marketplace subscription** to Kotoba TTS

### Required IAM Permissions

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:DeleteEndpoint",
                "sagemaker:DeleteEndpointConfig",
                "sagemaker:DescribeEndpoint",
                "sagemaker:InvokeEndpoint",
                "sagemaker:InvokeEndpointWithResponseStream"
            ],
            "Resource": "*"
        }
    ]
}
```

## Table of Contents

1. [Subscribe to Model Package](#1.-Subscribe-to-Model-Package)
2. [Setup](#2.-Setup)
3. [Deploy Real-time Endpoint](#3.-Deploy-Real-time-Endpoint)
4. [Perform Real-time Synthesis](#4.-Perform-Real-time-Synthesis)
5. [Pipelining & Cancellation (Queue Semantics)](#5.-Pipelining-&-Cancellation-(Queue-Semantics))
6. [Troubleshooting](#6.-Troubleshooting)
7. [Clean-up](#7.-Clean-up)

## 1. Subscribe to Model Package

To subscribe to the Kotoba TTS model package:

1. Open the [Kotoba TTS listing on AWS Marketplace](#) <!-- TODO: Update with actual URL -->
2. Click **Continue to Subscribe**
3. Review the pricing and EULA, then click **Subscribe**
4. Once subscribed, click **Continue to Configuration**
5. Select your preferred region and note the **Model Package ARN**

## 2. Setup

### 2.1 Install Dependencies

Run this cell first — the Python version check in 2.2 doesn't require these packages, but every cell after that does.

In [ ]:
!pip install -q --upgrade boto3 numpy
!pip install -q --upgrade "sagemaker>=2.0,<3"
!pip install -q --upgrade "aws-sdk-python[sagemaker-runtime-http2]"


### 2.2 Check Python Version

`aws-sdk-python` requires Python 3.12+. If this cell raises, switch your kernel (or create a new environment) before continuing.

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required for SageMaker bidirectional streaming. "
        f"Current version: {sys.version}"
    )

print(f"Python version: {sys.version}")


### 2.3 Import Libraries

In [ ]:
import asyncio
import base64
import json
import time
import wave

import boto3
import numpy as np
import sagemaker

# SageMaker HTTP/2 SDK for bidirectional streaming
from aws_sdk_sagemaker_runtime_http2.client import SageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.config import Config, HTTPAuthSchemeResolver
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from sagemaker import ModelPackage
from smithy_aws_core.auth.sigv4 import SigV4AuthScheme
from smithy_aws_core.identity import StaticCredentialsResolver

print("All dependencies imported successfully.")

### 2.4 Configure AWS Session

SageMaker needs an **execution role** (an IAM *role*, not an IAM *user*) to assume when creating the endpoint.

- **On SageMaker Studio / Notebook Instance**: leave `SAGEMAKER_ROLE_ARN = None`. `sagemaker.get_execution_role()` will pick up the attached role automatically.
- **Running locally (IAM user / CLI profile)**: set `SAGEMAKER_ROLE_ARN` to the ARN of a role whose trust policy allows `sagemaker.amazonaws.com` to assume it and which has (at minimum) the `AmazonSageMakerFullAccess` managed policy.


In [ ]:
# --- Execution role -----------------------------------------------------
# If running on SageMaker Studio or a Notebook Instance, leave this as None.
# If running locally as an IAM user, set it to the ARN of a SageMaker-capable role, e.g.
#   SAGEMAKER_ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
SAGEMAKER_ROLE_ARN: str | None = None
# ------------------------------------------------------------------------

sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name

if SAGEMAKER_ROLE_ARN is not None:
    role = SAGEMAKER_ROLE_ARN
else:
    try:
        role = sagemaker.get_execution_role()
    except ValueError as exc:
        raise ValueError(
            "get_execution_role() only works in SageMaker-managed environments. "
            "Set SAGEMAKER_ROLE_ARN above to a SageMaker execution role ARN "
            "(not your IAM user ARN). See the markdown cell above for role requirements."
        ) from exc

print(f"Region: {region}")
print(f"Role:   {role}")


### 2.5 Configure Model Package ARN

Replace `<MODEL_PACKAGE_ARN>` with the ARN from your AWS Marketplace subscription.

In [ ]:
# TODO: Replace with your Model Package ARN from AWS Marketplace
MODEL_PACKAGE_ARN = "<MODEL_PACKAGE_ARN>"

if MODEL_PACKAGE_ARN.startswith("<"):
    raise ValueError(
        "Please replace MODEL_PACKAGE_ARN with your actual Model Package ARN "
        "from AWS Marketplace subscription."
    )


## 3. Deploy Real-time Endpoint

### 3.1 Create Endpoint

Supported instance types:

| Instance | vCPU | Memory | GPU | Notes |
|----------|------|--------|-----|-------|
| `ml.g6e.8xlarge` | 32 | 256 GiB | 1x NVIDIA L40S | Default, good balance of cost and throughput |
| `ml.g6e.4xlarge` | 16 | 128 GiB | 1x NVIDIA L40S | Choose for cheaper instance                  |


In [ ]:
# Endpoint configuration
ENDPOINT_NAME = f"kotoba-tts-{int(time.time())}"
INSTANCE_TYPE = "ml.g6.2xlarge"  # or "ml.g6.4xlarge"

print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"Instance type: {INSTANCE_TYPE}")


In [ ]:
# Create Model from Model Package
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

# Deploy endpoint (this takes 5-10 minutes)
print(f"Deploying endpoint '{ENDPOINT_NAME}'...")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint '{ENDPOINT_NAME}' is InService.")


## 4. Perform Real-time Synthesis

### 4.1 Connection Protocol

Kotoba TTS uses SageMaker bidirectional streaming:

```
Client (HTTP/2 SDK) -> SageMaker Runtime endpoint (bidirectional stream)
```

**Note**: Standard WebSocket libraries cannot connect directly to SageMaker endpoints.

### 4.2 Output Audio Format

The output encoding is negotiated on the `open` event via the optional
`format` and `sample_rate` fields. Both default to `pcm_f32` @ 24000 Hz (the
historical behaviour), so existing clients are unaffected. The negotiated
values are echoed on `session.created`. Audio is base64-encoded inside
`audio.chunk.audio`.

| `format`  | Sample rate (`sample_rate`) | Channels | Encoding                  |
| --------- | --------------------------- | -------- | ------------------------- |
| `pcm_f32` | 8000 / 16000 / 24000 Hz     | 1 (mono) | Little-endian float32     |
| `pcm_16`  | 8000 / 16000 / 24000 Hz     | 1 (mono) | Little-endian signed 16-bit |
| `mulaw`   | 8000 Hz (fixed)             | 1 (mono) | 8-bit G.711 mu-law        |
| `opus`    | 24000 Hz (fixed)            | 1 (mono) | Ogg/Opus                  |

`mulaw` always emits at 8000 Hz and `opus` at 24000 Hz; a `sample_rate`
requested alongside them is ignored. An unsupported `format` or `sample_rate`
fails the `open` and closes the connection.

### 4.3 Protocol Flow

The client must send `open` first; the server replies with `session.created`
once the language and speaker are validated. If `open` is not sent within
`WS_OPEN_TIMEOUT` (default 10s), the server closes the connection.

```
Client                                    Server
  |---- open ------------------------------>|
  |     (language, speaker_id,              |
  |      format?, sample_rate?)             |
  |<---- session.created -------------------|
  |     (format, sample_rate, channels)     |
  |---- response.create ------------------->|
  |     (text, response_id?)                |
  |<---- response.created ------------------|
  |     (response.id)                       |
  |<---- audio.chunk (isFinal:false) -------|
  |     (response_id, base64 audio)         |
  |              ...                        |
  |<---- audio.chunk (isFinal:true) --------|
  |<---- response.done ---------------------|
  |     (response.id, status: completed)    |
```

### 4.4 Event Reference

#### Client Events (you send)

| Event              | Description                                                                                                                                         |
| ------------------ | -------------------------------------------------------------------------------------------------------------------------------------------------- |
| `open`             | First message; configure `language`, `speaker_id`, optional `format` / `sample_rate` (see 4.2)                                                     |
| `response.create`  | Submit the full text for synthesis. Required: `text`. Optional: `response_id`, `max_length`. **Repeatable without waiting** — extra ones queue (see §5). |
| `response.cancel`  | Cancel a response. `{"type":"response.cancel"}` cancels the **active** one; add `"response_id"` to cancel a specific **active or queued** one (see §5). |

> This model has **no `input_text_buffer.append` or `input_text_buffer.commit`** —
> the full text ships in `response.create`. The v0.1 input-streaming protocol
> is documented at [docs/tts-v0.1/](../tts-v0.1/).

#### Server Events (you receive)

| Event              | Description                                                                                                  |
| ------------------ | ------------------------------------------------------------------------------------------------------------ |
| `session.created`  | Session established (`format`, `sample_rate`, `channels`, `language`, `speaker_id`, `client_id`)             |
| `response.queued`  | A `response.create` was accepted but cannot start yet (another response is active). Payload: `response: {id}`. It will later emit `response.created`. **Only appears when pipelining** (see §5). |
| `response.created` | Synthesis turn started. Payload: `response: {id}` (the id supplied in `response.create`, or a server UUID).  |
| `audio.chunk`      | Base64 audio delta. Payload: `response_id`, `audio` (base64 in the negotiated `format`), `isFinal` (true marks the last chunk). |
| `response.done`    | Turn finished. Payload: `response: {id, status, error?, usage?}` where `status` is `completed`/`cancelled`/`failed`. `usage` = `{input_tokens, output_tokens, total_tokens}`, present when tokens were consumed. |
| `error`            | Error notification. Carries **`fatal`** (`true` ⇒ the connection is closing, reconnect; `false` ⇒ the session continues) and, for request-scoped errors, the **`response_id`** it concerns. |
| `timeout`          | Result timeout — the server produced no audio in time (non-fatal; session still alive)                                          |

**Per-response lifecycle** — every event below carries the same `response.id`. Two
branches surprise clients that assume "`response.created` always comes first":

```
response.create ─► response.created ─► audio.chunk* ─► response.done(completed | failed)
                                  └──► response.done(cancelled)        # cancelled while active
               └─► response.queued ─► response.created ─► … ─► response.done
                                  └──► response.done(cancelled)        # cancelled while queued — NO response.created
               └─► error(fatal:false, response_id)                     # rejected at submission (empty text, duplicate id, queue full)
```

§5 below demonstrates each of these paths against a live endpoint. See
[docs/tts/data/sample_input.json](./data/sample_input.json) and
[docs/tts/data/sample_output.json](./data/sample_output.json) for canonical payloads.

### 4.5 Prepare Input

`SPEAKER_ID` selects one of the available voice presets. `default` is always
available; additional preset keys depend on the bundle.

This model is one-shot: assemble the full text and send it inside a single
`response.create`. Sentence-level chunking is not required.

In [ ]:
SAMPLE_RATE = 24000
LANGUAGE = "ja"

# Select an available preset voice. "default" is always shipped. Other preset
# keys (e.g. "man-1", "woman-1") depend on the bundle.
SPEAKER_ID = "default"

# Full text to synthesize. This model is one-shot: the entire string is sent inside
# a single response.create event.
TEXT = "こんにちは、世界。今日もよろしくお願いします。"

# Optional. When provided, echoed back on response.created / audio.chunk /
# response.done so the client can correlate streams. Omit to let the server
# assign a UUID.
RESPONSE_ID = "resp_001"

print(f"Language:    {LANGUAGE}")
print(f"Speaker:     {SPEAKER_ID}")
print(f"Response id: {RESPONSE_ID}")
print(f"Text:        {TEXT!r} ({len(TEXT)} chars)")


### 4.6 Streaming Inference

In [ ]:
async def synthesize_text(
    endpoint_name: str,
    text: str,
    language: str = "ja",
    speaker_id: str = "default",
    response_id: str | None = None,
) -> bytes:
    """
    Synthesize speech using Kotoba TTS (promptless, one-shot) bidirectional streaming.

    Args:
        endpoint_name: SageMaker endpoint name.
        text: full text to synthesize. Sent inside a single response.create
            event; this model does not stream text chunks.
        language: target language (must be in the endpoint's
            `SUPPORTED_LANGUAGES`).
        speaker_id: preset voice identifier. `default` is always available;
            additional preset keys depend on the bundle.
        response_id: optional client-supplied response id. Echoed on
            response.created / audio.chunk / response.done. When omitted the
            server assigns a UUID.

    Returns:
        Raw audio bytes (pcm_f32, 24000 Hz, mono).
    """
    # Get AWS credentials
    session = boto3.Session()
    credentials = session.get_credentials().get_frozen_credentials()

    # Configure HTTP/2 client
    config = Config(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_access_key_id=credentials.access_key,
        aws_secret_access_key=credentials.secret_key,
        aws_session_token=credentials.token,
        aws_credentials_identity_resolver=StaticCredentialsResolver(),
        auth_scheme_resolver=HTTPAuthSchemeResolver(),
        auth_schemes={"aws.auth#sigv4": SigV4AuthScheme(service="sagemaker")},
    )

    client = SageMakerRuntimeHTTP2Client(config=config)

    # Start bidirectional stream
    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name,
            model_invocation_path="invocations-bidirectional-stream",
        )
    )

    async def send_event(payload: dict):
        body = json.dumps(payload).encode("utf-8")
        # NOTE: data_type="UTF8" is required so SageMaker forwards the frame
        # as text. Without it the server cannot
        # decode the payload and the connection fails before any handler runs.
        request = RequestStreamEventPayloadPart(
            value=RequestPayloadPart(bytes_=body, data_type="UTF8")
        )
        await stream.input_stream.send(request)

    async def recv_event() -> dict:
        message = await output_stream.receive()
        return json.loads(message.value.bytes_.decode())

    output = await stream.await_output()
    output_stream = output[1] if isinstance(output, tuple) else output

    # 1) Send open. The server waits for this within WS_OPEN_TIMEOUT (~10s)
    #    and replies with session.created once language/speaker are validated.
    await send_event({
        "type": "open",
        "language": language,
        "speaker_id": speaker_id,
    })

    # 2) Receive session.created
    data = await recv_event()
    assert data["type"] == "session.created", data
    print(
        f"Session created: format={data.get('format')} sr={data.get('sample_rate')} "
        f"ch={data.get('channels')} speaker_id={data.get('speaker_id')}"
    )

    # 3) Submit the full text in a single response.create
    create_payload: dict = {"type": "response.create", "text": text}
    if response_id is not None:
        create_payload["response_id"] = response_id
    await send_event(create_payload)

    # 4) Receive response.created and capture the server-assigned id (if the
    #    client did not supply one).
    data = await recv_event()
    assert data["type"] == "response.created", data
    server_response_id = (data.get("response") or {}).get("id")
    print(f"Response started: id={server_response_id}")

    # 5) Collect audio.chunk deltas until response.done.
    audio_buffer = bytearray()
    while True:
        data = await recv_event()
        event_type = data["type"]
        if event_type == "audio.chunk":
            audio_b64 = data.get("audio", "")
            is_final = data.get("isFinal", False)
            audio_buffer.extend(base64.b64decode(audio_b64))
            print(
                f"Got audio chunk: response_id={data.get('response_id')} "
                f"b64_chars={len(audio_b64)} isFinal={is_final}"
            )
            continue
        if event_type == "response.done":
            response = data.get("response", {})
            status = response.get("status")
            print(f"Response done: id={response.get('id')} status={status} usage={response.get('usage')}")
            if status != "completed":
                raise RuntimeError(f"Synthesis failed: {data}")
            break
        if event_type in ("error", "timeout"):
            # `timeout` is non-fatal on its own; after WS_RESULT_TIMEOUT_LIMIT
            # consecutive timeouts the server emits a fatal `error` and closes.
            raise RuntimeError(f"Server event: {data}")

    await stream.input_stream.close()
    return bytes(audio_buffer)


In [ ]:
# Run synthesis
print("Starting synthesis...\n")

audio_bytes = await synthesize_text(
    endpoint_name=ENDPOINT_NAME,
    text=TEXT,
    language=LANGUAGE,
    speaker_id=SPEAKER_ID,
    response_id=RESPONSE_ID,
)

print(f"\nReceived {len(audio_bytes)} bytes of pcm_f32 audio")

# Convert pcm_f32 -> int16 and save as WAV for easy playback
samples_f32 = np.frombuffer(audio_bytes, dtype=np.float32)
samples_i16 = np.clip(samples_f32, -1.0, 1.0) * 32767.0
samples_i16 = samples_i16.astype(np.int16)

out_path = "sample_output.wav"
with wave.open(out_path, "wb") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(SAMPLE_RATE)
    w.writeframes(samples_i16.tobytes())

print(f"Saved synthesized audio to {out_path} ({len(samples_i16) / SAMPLE_RATE:.2f} s)")


## 5. Pipelining & Cancellation (Queue Semantics)

§4 sent one `response.create` and waited for `response.done`. But a session can
hold **many** responses in flight: you may send the next `response.create`
without waiting, and the server **queues** them and synthesizes one at a time in
FIFO order. The queue and cancel rules are the subtlest part of the protocol, so
this section both explains them and **proves each one against your live
endpoint** — every demo prints the exact event timeline it received and asserts
the ordering.

### The rules

- **One at a time, FIFO.** If no response is active, `response.create` starts now
  (`response.created`). If a response is already active, the new one is accepted
  with **`response.queued`** and starts later (`response.created`) when its turn
  comes. `response.done(N)` is always delivered **before** `response.created(N+1)`.
- **Cancel by id.** `response.cancel` with a `response_id` cancels that response —
  whether it is **active** (stops synthesis, reports any `usage` already consumed, and promotes
  the next queued one) or still **queued** (drops it with **no** `response.created`
  and **no** `usage`). Without an id it cancels whatever is active; in a pipelined
  client, **always cancel by id** — "what's active right now" is ambiguous when
  several responses are outstanding.
- **Unique ids.** Each outstanding `response_id` must be unique. Reusing an
  active/queued id is rejected with a non-fatal `error`. Reuse is fine once its
  `response.done` has been delivered.
- **Bounded queue.** At most `WS_MAX_PENDING_RESPONSES` (default 15) may wait;
  beyond that `response.create` is rejected with a non-fatal `error`.
- **Correlated, classified errors.** A request-scoped `error` carries the
  `response_id` it concerns and `fatal: false` — the session keeps going.

### Two traps (see the lifecycle diagram in §4.4)

1. A **queued** response that is cancelled jumps straight to
   `response.done(cancelled)` — there is **no** `response.created` for it.
2. A `response.create` **rejected at submission** ends with an id-correlated
   `error` **instead of** a `response.done`.

### What each demo proves

| Demo | Proves |
| ---- | ------ |
| 5.1 Sequential | Baseline: wait-for-done ⇒ no `response.queued` |
| 5.2 Pipelining | `response.queued(B)` precedes `response.created(B)`; `done(A)` precedes `created(B)` |
| 5.3 FIFO | Queued responses start in send order (A → B → C) |
| 5.4 Cancel active | `done(cancelled A)`, then the queued **B is promoted automatically** |
| 5.5 Cancel queued | The cancelled queued **B never starts** (no `response.created`); C still runs |
| 5.6 Errors | Rejections are **non-fatal** and carry the offending **`response_id`** |

> **Timing note.** The queue/cancel demos rely on the first response (`A`,
> `LONG_TEXT`) still synthesizing while later events are sent. If your endpoint is
> very fast and `A` finishes early, the affected demo prints a "re-run with a
> longer `LONG_TEXT`" hint instead of failing — just lengthen `LONG_TEXT` in the
> helper cell and re-run.

In [ ]:
# ── Reusable teaching client + helpers ───────────────────────────────────────
# TTSDemoSession opens ONE bidirectional stream and records EVERY received event
# in arrival order, so each demo below can print the timeline and assert the
# ordering. (Run cells 2.x, 3.x first so `region` and `ENDPOINT_NAME` exist.)
import time as _time

# A keeps the server busy while later events are sent; B/C are short. If a
# cancel/queue demo says "re-run with a longer LONG_TEXT", lengthen this.
LONG_TEXT = (
    "これはキューイングとキャンセルの挙動を確認するための、十分に長い読み上げ用テキストです。"
    "後続のリクエストを送っている間、最初の応答が合成を続けている必要があります。"
    "よろしくお願いいたします。"
)
SHORT_TEXT = "こんにちは。"


def etype(e):
    return e.get("type")


def eid(e):
    """response.created/queued/done carry response.id; audio.chunk & error carry response_id."""
    r = e.get("response")
    if isinstance(r, dict) and r.get("id") is not None:
        return r["id"]
    return e.get("response_id")


def estatus(e):
    r = e.get("response")
    return r.get("status") if isinstance(r, dict) else None


def first(events, type_, rid=None):
    """Index of the first event of `type_` (and id `rid`, if given), or None."""
    for i, e in enumerate(events):
        if etype(e) == type_ and (rid is None or eid(e) == rid):
            return i
    return None


def saw(events, type_, rid=None):
    return first(events, type_, rid) is not None


def done_status(events, rid):
    i = first(events, "response.done", rid)
    return estatus(events[i]) if i is not None else None


def done(rid):
    """A `collect` stop predicate: True once response.done for `rid` has arrived."""
    return lambda events: saw(events, "response.done", rid)


def show_timeline(events, title=""):
    """Print every received event in arrival order — the heart of these demos."""
    if title:
        print(title)
    for i, e in enumerate(events):
        t = etype(e)
        if t == "response.done":
            u = (e.get("response") or {}).get("usage")
            extra = f"status={estatus(e)}" + (f" usage={u}" if u else "")
        elif t == "audio.chunk":
            extra = f"isFinal={e.get('isFinal')}"
        elif t == "error":
            extra = f"fatal={e.get('fatal')} msg={e.get('message')!r}"
        else:
            extra = ""
        print(f"  {i:2}. {t:18} id={str(eid(e)):16} {extra}")


def check(cond, msg):
    print(f"  {'OK  ' if cond else 'FAIL'}: {msg}")
    assert cond, msg


def assert_before(events, a, b, msg):
    """Assert the first event matching a=(type, id) precedes the first matching b."""
    ia, ib = first(events, *a), first(events, *b)
    check(ia is not None and ib is not None and ia < ib, msg)


class TTSDemoSession:
    """One connection. Send open / response.create / response.cancel; `collect`
    records every received event in order so demos can show and assert behaviour."""

    def __init__(self, endpoint_name, language="ja", speaker_id="default"):
        self.endpoint_name = endpoint_name
        self.language = language
        self.speaker_id = speaker_id
        self.events = []
        self._stream = None
        self._output = None

    async def __aenter__(self):
        creds = boto3.Session().get_credentials().get_frozen_credentials()
        config = Config(
            endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
            region=region,
            aws_access_key_id=creds.access_key,
            aws_secret_access_key=creds.secret_key,
            aws_session_token=creds.token,
            aws_credentials_identity_resolver=StaticCredentialsResolver(),
            auth_scheme_resolver=HTTPAuthSchemeResolver(),
            auth_schemes={"aws.auth#sigv4": SigV4AuthScheme(service="sagemaker")},
        )
        client = SageMakerRuntimeHTTP2Client(config=config)
        self._stream = await client.invoke_endpoint_with_bidirectional_stream(
            InvokeEndpointWithBidirectionalStreamInput(
                endpoint_name=self.endpoint_name,
                model_invocation_path="invocations-bidirectional-stream",
            )
        )
        out = await self._stream.await_output()
        self._output = out[1] if isinstance(out, tuple) else out
        return self

    async def __aexit__(self, *exc):
        try:
            await self._stream.input_stream.close()
        except Exception:
            pass

    async def _send(self, payload):
        body = json.dumps(payload).encode("utf-8")
        await self._stream.input_stream.send(
            RequestStreamEventPayloadPart(value=RequestPayloadPart(bytes_=body, data_type="UTF8"))
        )

    async def open(self):
        await self._send({"type": "open", "language": self.language, "speaker_id": self.speaker_id})
        msg = await self._output.receive()
        ev = json.loads(msg.value.bytes_.decode())
        assert etype(ev) == "session.created", ev
        return ev

    async def create(self, text, response_id=None, max_length=None):
        payload = {"type": "response.create", "text": text}
        if response_id is not None:
            payload["response_id"] = response_id
        if max_length is not None:
            payload["max_length"] = max_length
        await self._send(payload)
        return response_id

    async def cancel(self, response_id=None):
        payload = {"type": "response.cancel"}
        if response_id is not None:
            payload["response_id"] = response_id
        await self._send(payload)

    async def collect(self, stop, *, timeout=120.0):
        """Read events into self.events (arrival order) until stop(events) is True."""
        deadline = _time.monotonic() + timeout
        while not stop(self.events):
            remaining = deadline - _time.monotonic()
            if remaining <= 0:
                raise TimeoutError(f"stop not met after {len(self.events)} events")
            msg = await asyncio.wait_for(self._output.receive(), timeout=remaining)
            payload = getattr(msg.value, "bytes_", None)
            if payload is not None:
                self.events.append(json.loads(payload.decode()))
        return self.events

    async def expect_error(self, *, timeout=60.0):
        """Read (without treating `error` as fatal) until an `error` arrives; return it."""
        deadline = _time.monotonic() + timeout
        while _time.monotonic() < deadline:
            remaining = deadline - _time.monotonic()
            msg = await asyncio.wait_for(self._output.receive(), timeout=remaining)
            payload = getattr(msg.value, "bytes_", None)
            if payload is None:
                continue
            ev = json.loads(payload.decode())
            self.events.append(ev)
            if etype(ev) == "error":
                return ev
        return None


print("Demo helpers ready: TTSDemoSession, show_timeline, check, assert_before, done(), saw(), first()")


In [ ]:
# ── 5.1 Sequential (baseline) ────────────────────────────────────────────────
# Wait for response.done before the next response.create. No response is active when each is
# submitted, so neither is ever queued (no response.queued appears).
async with TTSDemoSession(ENDPOINT_NAME) as s:
    await s.open()
    await s.create(SHORT_TEXT, response_id="A")
    await s.collect(done("A"))
    await s.create(SHORT_TEXT, response_id="B")
    await s.collect(done("B"))

show_timeline(s.events, "5.1 Sequential — created→done, then created→done:")
check(not saw(s.events, "response.queued"), "no response.queued (nothing was queued)")
check(done_status(s.events, "A") == "completed", "A completed")
check(done_status(s.events, "B") == "completed", "B completed")


In [ ]:
# ── 5.2 Pipelining (response.queued) ─────────────────────────────────────────
# Send A (long) then B back-to-back. While A is active B cannot start, so it is
# acknowledged with response.queued and only later gets response.created — and
# response.done(A) always arrives before response.created(B).
async with TTSDemoSession(ENDPOINT_NAME) as s:
    await s.open()
    await s.create(LONG_TEXT, response_id="A")   # starts immediately
    await s.create(SHORT_TEXT, response_id="B")  # queued behind A
    await s.collect(done("B"))

show_timeline(s.events, "5.2 Pipelining — A and B sent back-to-back:")
if not saw(s.events, "response.queued", "B"):
    print("  (A finished before B was submitted, so B started immediately — re-run with a longer LONG_TEXT)")
else:
    assert_before(s.events, ("response.queued", "B"), ("response.created", "B"), "queued(B) before created(B)")
    assert_before(s.events, ("response.done", "A"), ("response.created", "B"), "done(A) before created(B)")
    check(done_status(s.events, "A") == "completed", "A completed")
    check(done_status(s.events, "B") == "completed", "B completed")


In [ ]:
# ── 5.3 FIFO ordering ────────────────────────────────────────────────────────
# Three responses sent at once drain in the order they were submitted (A→B→C).
# Each one's done precedes the next one's created.
async with TTSDemoSession(ENDPOINT_NAME) as s:
    await s.open()
    await s.create(LONG_TEXT, response_id="A")
    await s.create(SHORT_TEXT, response_id="B")
    await s.create(SHORT_TEXT, response_id="C")
    await s.collect(done("C"))

show_timeline(s.events, "5.3 FIFO — A, B, C sent back-to-back:")
if not (saw(s.events, "response.queued", "B") and saw(s.events, "response.queued", "C")):
    print("  (A finished before B/C were submitted, so not all were queued — re-run with a longer LONG_TEXT)")
else:
    assert_before(s.events, ("response.created", "A"), ("response.created", "B"), "created A before B")
    assert_before(s.events, ("response.created", "B"), ("response.created", "C"), "created B before C (FIFO)")
    assert_before(s.events, ("response.done", "A"), ("response.created", "B"), "done(A) before created(B)")
    assert_before(s.events, ("response.done", "B"), ("response.created", "C"), "done(B) before created(C)")
    check(done_status(s.events, "C") == "completed", "C completed")


In [ ]:
# ── 5.4 Cancel the active response (the next queued one is promoted) ──────────
# A is running, B is queued. Cancelling A (by id) ends A with status=cancelled,
# and the server automatically promotes B: you see done(cancelled A) then
# created(B). (Cancel-by-id also works for the active response.)
async with TTSDemoSession(ENDPOINT_NAME) as s:
    await s.open()
    await s.create(LONG_TEXT, response_id="A")
    await s.create(SHORT_TEXT, response_id="B")   # queued behind A
    await s.collect(lambda e: saw(e, "response.created", "A") and saw(e, "response.queued", "B"))
    await s.cancel(response_id="A")               # cancel the ACTIVE one (by id)
    await s.collect(done("B"))

show_timeline(s.events, "5.4 Cancel active A; B promoted:")
if done_status(s.events, "A") == "completed":
    print("  (A finished before the cancel — re-run with a longer LONG_TEXT)")
else:
    check(done_status(s.events, "A") == "cancelled", "A cancelled")
    assert_before(s.events, ("response.done", "A"), ("response.created", "B"), "done(cancelled A) before created(B)")
    check(done_status(s.events, "B") == "completed", "promoted B completed")


In [ ]:
# ── 5.5 Cancel a queued response by id ───────────────────────────────────────
# A is running; B and C are queued. Cancelling B (still queued) drops it from the
# queue WITHOUT ever emitting response.created for it (and with no usage); C is
# unaffected and runs after A. This is trap #1: queued cancel -> done(cancelled).
async with TTSDemoSession(ENDPOINT_NAME) as s:
    await s.open()
    await s.create(LONG_TEXT, response_id="A")
    await s.create(SHORT_TEXT, response_id="B")   # queued, will be cancelled
    await s.create(SHORT_TEXT, response_id="C")   # queued, should still run
    await s.collect(lambda e: saw(e, "response.queued", "B") and saw(e, "response.queued", "C"))
    await s.cancel(response_id="B")               # cancel the QUEUED one by id
    await s.collect(done("C"))

show_timeline(s.events, "5.5 Cancel queued B by id; C still runs:")
ia, ib = first(s.events, "response.done", "A"), first(s.events, "response.created", "B")
if ib is not None and ia is not None and ia < ib:
    print("  (A finished before the cancel — B was promoted then cancelled; re-run with a longer LONG_TEXT)")
else:
    check(done_status(s.events, "B") == "cancelled", "queued B cancelled")
    check(not saw(s.events, "response.created", "B"), "B never started (no response.created for B)")
    i_b = first(s.events, "response.done", "B")
    check((s.events[i_b].get("response") or {}).get("usage") is None, "queued-cancel B carries no usage")
    check(saw(s.events, "response.created", "C"), "C started")
    check(done_status(s.events, "C") == "completed", "C completed")


In [ ]:
# ── 5.6 Error correlation & fatal flag ───────────────────────────────────────
# Request-scoped rejections come back as a NON-FATAL error carrying the offending
# response_id (so a pipelined client knows which create/cancel failed); the
# session stays open. (A fatal error, e.g. an invalid `open`, would set
# fatal=true and the connection would close.)
async def show_error(label, drive, allow_missing=False):
    async with TTSDemoSession(ENDPOINT_NAME) as s:
        await s.open()
        await drive(s)
        err = await s.expect_error()
        print(f"  {label:18} -> {err}")
        if err is None and allow_missing:
            return None
        check(err is not None, f"{label}: error received")
        check(err is not None and err.get("fatal") is False, f"{label}: non-fatal")
        return err

# (a) empty text — rejected, correlated to the id you sent
err = await show_error("empty text", lambda s: s.create("", response_id="bad-text"))
check(eid(err) == "bad-text", "empty-text error.response_id == 'bad-text'")


# (b) duplicate response_id while one is still outstanding
async def _dup(s):
    await s.create(LONG_TEXT, response_id="dup")   # active, holds the id
    await s.create(SHORT_TEXT, response_id="dup")  # reuse -> rejected
err = await show_error("duplicate id", _dup, allow_missing=True)
if err is None:
    print("  (first 'dup' finished before the second was submitted — no collision; re-run with a longer LONG_TEXT)")
else:
    check(eid(err) == "dup", "duplicate error.response_id == 'dup'")

# (c) cancel an id that does not exist
err = await show_error("cancel unknown id", lambda s: s.cancel(response_id="ghost"))
check(eid(err) == "ghost", "cancel-not-found error.response_id == 'ghost'")


## 6. Troubleshooting

| Symptom | Likely cause & fix |
|---------|--------------------|
| `RuntimeError: Python 3.12+ is required` | The HTTP/2 SDK requires Python 3.12+. Switch your kernel or create a new conda/venv with Python 3.12. |
| `ModuleNotFoundError: aws_sdk_sagemaker_runtime_http2` | Re-run the install cell in 2.1 (`pip install aws-sdk-python[sagemaker-runtime-http2]`). The package name uses underscores when imported. |
| Connection closes immediately after invoke (no `session.created`) | The first event must be `open` and must be sent within `WS_OPEN_TIMEOUT` (default 10s). Make sure `RequestPayloadPart` is constructed with `data_type="UTF8"` so SageMaker forwards a text frame. |
| Server replies with `error` / `unsupported_language` | The endpoint's `SUPPORTED_LANGUAGES` does not include the requested language. Use `language="ja"`. This error is **fatal** (`fatal: true`) and closes the connection. |
| Server replies with `error` / `invalid message type` after sending `input_text_buffer.append` | You are using the v0.1 input-streaming protocol against a v1.x endpoint. This model is one-shot: put the full text inside `response.create` instead. See [docs/tts-v0.1/](../tts-v0.1/) for the v0.1 protocol. |
| `response.done` with `status: failed` and `unknown_speaker_id` | The requested `speaker_id` is not one of the available presets. Use `"default"` (always available) or another preset key shipped with the bundle. |
| `error` / `text must be a non-empty string` (`fatal: false`) | `response.create` requires a non-empty `text` field. The error carries the `response_id` you sent; the session stays open, so just resend a corrected `response.create`. |
| Second `response.create` returns `response.queued`, not `response.created` | **Expected** — you pipelined a request while another was active. It will emit `response.created` when its turn comes (FIFO). See §5. |
| `error` / `response_id already in use` (`fatal: false`) | That id is still attached to an active or queued response. Use a fresh `response_id`, or reuse it only after that response's `response.done`. |
| `error` / `too many queued responses` (`fatal: false`) | The pending queue is full (`WS_MAX_PENDING_RESPONSES`, default 15). Let some `response.done`s drain before queueing more. |
| `error` / `no active response` or `response not found` on `response.cancel` (`fatal: false`) | The cancel target already finished or never existed — typically a cancel that raced a natural completion. Harmless; the error's `response_id` tells you which one. Prefer cancelling **by `response_id`** in pipelined clients. |
| `response.done(cancelled)` for a queued id with no preceding `response.created` | **Expected** — a queued response cancelled before it started has no `response.created`. See the lifecycle diagram in §4.4. |
| `timeout` event during synthesis | The server did not produce audio within `WS_RESULT_TIMEOUT` (default 60s). Non-fatal — usually transient; retry. After `WS_RESULT_TIMEOUT_LIMIT` (default 3) consecutive timeouts the server escalates to a **fatal** `error` (`fatal: true`) and closes. |
| Idle connection closed after ~60s | Hit `WS_IDLE_TIMEOUT`. The timer is reset by client messages and major server events (not `audio.chunk`), so draining a queue keeps it alive; the countdown effectively starts after the last `response.done`. Send any client event before the timeout, or open a fresh connection per turn. |
| `AccessDeniedException` at `model.deploy(...)` | The execution role cannot pull the Marketplace model. Verify the AWS Marketplace subscription is active and that the role has `AmazonSageMakerFullAccess` (or equivalent). |
| Saved WAV is silent / sounds like white noise | The pcm_f32 -> int16 conversion was skipped or the byte order is wrong. The server emits little-endian float32; decode with `np.frombuffer(..., dtype=np.float32)` before scaling. |

## 7. Clean-up

**Important**: Delete the endpoint when done. GPU instances (`ml.g6e.8xlarge` / `ml.g6e.4xlarge`) are billed **per hour while the endpoint is active**, even when idle.

In [ ]:
print(f"Deleting endpoint '{ENDPOINT_NAME}'...")

sagemaker_session.delete_endpoint(ENDPOINT_NAME)
sagemaker_session.delete_endpoint_config(ENDPOINT_NAME)

print("Endpoint deleted.")


### Unsubscribe from Model Package (Optional)

To unsubscribe from the Kotoba TTS model package:

1. Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)
2. Find "Kotoba TTS" in your subscriptions
3. Click **Manage** -> **Actions** -> **Cancel subscription**

**Note**: You must delete all endpoints using this model package before unsubscribing.

---

## Support

For technical support, please contact Kotoba AI support.

## References

- Canonical protocol samples: [docs/tts/data/sample_input.json](./data/sample_input.json), [docs/tts/data/sample_output.json](./data/sample_output.json), [docs/tts/data/README.md](./data/README.md)
- Previous version docs (input-streaming protocol): [docs/tts-v0.1/](../tts-v0.1/)
- [SageMaker Bidirectional Streaming Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-test-endpoints.html)
- [AWS SDK for Python (Boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)